# **From Raw Table to Analysis Dataset: Auditing and Preparing 500,000 Georgia Mortgage Records**

**How a public lending table becomes a dataset that can be trusted:**
- What does the table contain?
- What is wrong with it?
- What is decided about each problem?
- What is built from it?

**Author:** Emely Lephilibert  
**Data:** Home Mortgage Disclosure Act (HMDA), FFIEC and CFPB Data Browser, Georgia, 2025, snapshot of 2026-11-09  
**Language:** Python  
**Notebook 1 of 2:** this notebook builds the analysis dataset. The second notebook, [What the Data Leaves Out](https://github.com/emelylephilibert/mortgage-missing-data-influence/blob/main/02_missing_incomes_and_extreme_loans.ipynb), analyses it.

## The problem

Every estimate built from lending data inherits the problems of the records behind it. A mortgage table can hold text codes in place of missing values, fields that some lenders are allowed not to report, incomes and loan amounts far outside the usual range, and values grouped into bands instead of numbers. If those problems are not found and decided one by one, they end up inside benchmarks and models, where nobody can see them.

This notebook starts from the raw HMDA records of mortgage applications in Georgia for 2025 and ends with a dataset that an analysis can use. It shows what the table contains, which problems it has, what is decided about each one and why, and which variables are built on top of it. **Here the missing and extreme values are not only cleaned: they are recorded and kept visible, because the second notebook measures how much they change the results.**

## What this notebook produces

- A **dated snapshot** of the source table, saved exactly as received.
- An **issues log** with every problem found in the audit and the decision taken for each one.
- A **processed dataset** with the cleaned columns, the flags that mark each decision, and the derived variables, with a data dictionary.

Two rules guide the work. No raw value is overwritten: where a value is corrected or set aside, the result goes into a new column or a flag. And every decision is written as code, so the processed dataset can be rebuilt from the snapshot at any time.

## The five chapters

| Chapter | Question | Why it matters |
|---------|----------|----------------|
| 1. The source | Where does the table come from, and is the copy complete? | An analysis can only be repeated if the data are frozen and checked. |
| 2. The audit | What does the table contain, and what is wrong with it? | Problems that are not named are not handled. |
| 3. The decisions | What is done about each problem? | A decision that is written down can be reviewed and reversed. |
| 4. The new variables | Which variables does the analysis need that the table does not have? | Each definition is made once, so every analysis uses the same one. |
| 5. The output | What is saved, and what does each column mean? | The dataset is only useful if it is documented. |

## How to read this notebook

Each chapter opens with its question, shows the evidence in tables, and closes with a **Summary** that gives the question, the answer and the evidence behind it. The published HMDA data can change as lenders correct and resubmit their filings, so the notebook downloads the table once, saves a dated snapshot, and every later step reads that snapshot. The code is included so that every number can be reproduced.

## Setup

The notebook uses pandas for the tables and requests to read the HMDA Data Browser. The raw snapshot and the processed dataset are saved in two folders, `data/raw` and `data/processed`.

In [ ]:
# Import the libraries: web requests, data tables and file handling
import requests
import pandas as pd
from pathlib import Path

# Address of the Data Browser endpoint that returns HMDA records as CSV
BASE = "https://ffiec.cfpb.gov/v2/data-browser-api/view/csv"

# Browser-like header: the server refuses requests that do not send one
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

# Folders for the snapshot and the processed dataset, relative to the repository root
RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Chapter 1. The source

**Question:** Where does the table come from, and is the copy complete?

**Why it matters:** An analysis can only be repeated if the data are frozen and checked. Lenders can correct and resubmit their filings after publication, so a copy that is not dated and verified cannot be compared with another one.

## 1.1 What is kept

The table has columns that the analysis does not need. 33 columns are kept: identifier and scope, outcome of the application, loan characteristics, pricing, amounts and borrower finances, and census tract context. The columns left out are the demographic fields of applicants and co-applicants (ethnicity, race, sex and age), which this project does not analyse, the reasons for denial, the automated underwriting systems, the credit score model, and less common loan features such as balloon or interest-only payments.

In [ ]:
# Columns kept in the snapshot, grouped by role
KEEP = [
    # identifier and scope
    "activity_year", "lei", "state_code", "county_code", "derived_msa-md",
    # outcome of the application
    "action_taken", "preapproval",
    # loan characteristics
    "loan_type", "loan_purpose", "lien_status", "occupancy_type", "open-end_line_of_credit",
    "business_or_commercial_purpose", "reverse_mortgage", "construction_method", "total_units",
    "derived_dwelling_category", "derived_loan_product_type", "conforming_loan_limit", "loan_term",
    # pricing
    "interest_rate", "rate_spread", "total_loan_costs", "origination_charges",
    "discount_points", "lender_credits",
    # amounts and borrower finances
    "loan_amount", "property_value", "loan_to_value_ratio", "income", "debt_to_income_ratio",
    # census tract context
    "tract_to_msa_income_percentage", "ffiec_msa_md_median_family_income",
]

# Check that the list has 33 columns and no repeats
print(len(KEEP), len(set(KEEP)))

33 33


## 1.2 The download

The Data Browser total for Georgia in 2025 is read before and after the download, and the two must match, so that the copy is a complete picture of the table at one moment. The Data Browser returns the full table with every column, so the download goes to a temporary file first. The notebook checks that every kept column exists, then reads only those columns. Every value is read as text and the codes "NA" and "Exempt" are kept as they are, so nothing is changed at this step.

The copy is saved as a dated file. If the file of that date already exists, the notebook loads it instead of downloading again, so every run works on the same table.

In [ ]:
# Scope of the snapshot and name of its file (change the date only to download a new snapshot)
STATE, YEAR = "GA", 2025
SNAP_DATE = "2026-11-09"
SNAP_FILE = RAW_DIR / f"hmda_{STATE}_{YEAR}_{SNAP_DATE}.csv.gz"

# Ask the Data Browser how many records the state has for the year, across the eight action types
def portal_total():
    r = requests.get("https://ffiec.cfpb.gov/v2/data-browser-api/view/aggregations",
                     params={"states": STATE, "years": YEAR, "actions_taken": "1,2,3,4,5,6,7,8"},
                     headers=HEADERS, timeout=120)
    r.raise_for_status()                                             # stop with an error if the server rejects the request
    return int(sum(a["count"] for a in r.json()["aggregations"]))    # add the counts of the eight action types

# Download the full table to a temporary file, check the kept columns, and return them as raw text
def download_snapshot(tmp_file=Path("hmda_download.csv")):
    with requests.get(BASE, params={"states": STATE, "years": YEAR},
                      headers=HEADERS, stream=True, timeout=900) as r:
        r.raise_for_status()                                         # stop with an error if the server rejects it
        with open(tmp_file, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):         # write the file in pieces of 1 MB
                f.write(chunk)
    header = pd.read_csv(tmp_file, nrows=0).columns                  # column names of the full table
    missing = [c for c in KEEP if c not in header]
    if missing:                                                      # stop if a kept column does not exist
        raise KeyError(f"Kept columns not found in the download: {missing}")
    print(f"Full table: {len(header)} columns")
    table = pd.read_csv(tmp_file, usecols=KEEP, dtype=str, keep_default_na=False)[KEEP]  # raw text, "NA" and "Exempt" kept
    tmp_file.unlink()                                                # delete the temporary file
    return table

In [ ]:
# Load the stored snapshot if it exists; otherwise download it and check the Data Browser total
if SNAP_FILE.exists():
    snap = pd.read_csv(SNAP_FILE, dtype=str, keep_default_na=False)
    print(f"Loaded the stored snapshot: {SNAP_FILE}")
else:
    total_before = portal_total()
    snap = download_snapshot()
    total_after = portal_total()
    print(f"Data Browser total before: {total_before:,}")
    print(f"Data Browser total after:  {total_after:,}")
    print("Table unchanged during the download:", total_before == total_after)

    # Save only a complete copy: the table did not change and every record was received
    if total_before != total_after or len(snap) != total_after:
        raise RuntimeError("The copy is not complete. Run the cell again.")
    snap.to_csv(SNAP_FILE, index=False)
    print(f"Saved: {SNAP_FILE}")

# Check the copy: size, and that every record belongs to the chosen state and year
print(f"Rows: {len(snap):,} | columns: {snap.shape[1]}")
print("States:", snap["state_code"].value_counts().to_dict())
print("Years: ", snap["activity_year"].value_counts().to_dict())

Full table: 99 columns
Data Browser total before: 515,746
Data Browser total after:  515,746
Table unchanged during the download: True
Saved: data/raw/hmda_GA_2025_2026-11-09.csv.gz
Rows: 515,746 | columns: 33
States: {'GA': 515746}
Years:  {'2025': 515746}


The full table has 99 columns, of which the 33 kept are saved. The snapshot has 515,746 rows, the same number as the Data Browser total read before and after the download, so the table did not change during the download and every record was received. All 515,746 records belong to Georgia and to 2025. The public HMDA table has no application identifier, so the completeness of the copy rests on the match with the Data Browser total. The file is saved only when that match holds, so the existence of the dated file means that the copy is a complete picture of the table on 2026-11-09.

## 1.3 Check of the saved file

The file is read back from disk, as text, and compared with the table in memory. The check also confirms that the raw text codes "NA" and "Exempt" were kept and not converted to missing values.

In [ ]:
# Read the saved file back as raw text, keeping "NA" and "Exempt" as they are
check = pd.read_csv(SNAP_FILE, dtype=str, keep_default_na=False)

# Compare the file with the table in memory
print("Same shape:     ", check.shape == snap.shape)
print("Same content:   ", check.equals(snap))

# Count the raw codes "NA" and "Exempt" in the debt-to-income column of both tables
for code in ["NA", "Exempt"]:
    print(f'"{code}" in memory: {(snap["debt_to_income_ratio"] == code).sum():,} | '
          f'in the file: {(check["debt_to_income_ratio"] == code).sum():,}')

Same shape:      True
Same content:    True
"NA" in memory: 204,247 | in the file: 204,247
"Exempt" in memory: 11,192 | in the file: 11,192


The saved file has the same shape and the same content as the table in memory. In the debt-to-income column, the 204,247 raw "NA" codes and the 11,192 raw "Exempt" codes were kept in both, so the snapshot is stored exactly as received from the Data Browser.

## Summary of Chapter 1: The source

**Question:** Where does the table come from, and is the copy complete?

**Answer:** The table is the Home Mortgage Disclosure Act (HMDA) data that the FFIEC and the CFPB publish through the HMDA Data Browser, for Georgia in 2025. The copy of 2026-11-09 is complete: 515,746 records with 33 of the 99 columns, saved exactly as received.

**What the evidence shows:**

- The copy is saved only if the Data Browser total, read before and after the download, is the same and equals the number of rows received (515,746). All records belong to Georgia and to 2025. The public table has no application identifier, so this match is the check of completeness.
- 33 columns are kept. The demographic fields of applicants and co-applicants, the reasons for denial, the automated underwriting systems, the credit score model and less common loan features are left out because no question needs them.
- The file read back from disk is identical to the table in memory, and the raw codes of the debt-to-income column (204,247 "NA" and 11,192 "Exempt") were kept, so nothing was changed at this step.
- The snapshot is dated and saved. Lenders can correct and resubmit their filings, so every later step reads this file and not the Data Browser, and every run works on the same table.

# Chapter 2. The audit

**Question:** What does the table contain, and what is wrong with it?

**Why it matters:** Problems that are not named are not handled. The audit changes nothing in the data: it counts, lists and describes, and it ends with a log of every problem found, which Chapter 3 uses to decide what to do with each one.

## 2.1 Structure

The snapshot is all text. A working table is made from it and the snapshot stays untouched. Before converting the numeric columns, the values that cannot be read as a number are listed, which shows how missing and withheld values are written in each column. The debt-to-income ratio and the number of units are stored in bands, such as "20%-<30%", so they stay as text and are audited as categories.

In [ ]:
# Working copy of the snapshot, the original table stays untouched
df = snap.copy()

# Columns meant to hold numbers
NUM_COLS = ["loan_amount", "property_value", "loan_to_value_ratio", "income",
            "interest_rate", "rate_spread", "total_loan_costs", "origination_charges",
            "discount_points", "lender_credits", "loan_term",
            "tract_to_msa_income_percentage", "ffiec_msa_md_median_family_income"]

# For each column, list the text values that cannot be read as a number, with their counts (top 10)
for col in NUM_COLS:
    values = df[col]
    parsed = pd.to_numeric(values, errors="coerce")
    bad = values[parsed.isna()].value_counts().head(10)          # values that failed to convert
    print(col, "->", bad.to_dict() if len(bad) else "all values convert")

loan_amount -> all values convert
property_value -> {'NA': 123811, 'Exempt': 11190}
loan_to_value_ratio -> {'NA': 201125, 'Exempt': 11191}
income -> {'NA': 92946}
interest_rate -> {'NA': 191436, 'Exempt': 11192}
rate_spread -> {'NA': 277487, 'Exempt': 11236}
total_loan_costs -> {'NA': 268826, 'Exempt': 11232}
origination_charges -> {'NA': 265849, 'Exempt': 11322}
discount_points -> {'NA': 270155, '': 96519, 'Exempt': 11322}
lender_credits -> {'NA': 269058, '': 148810, 'Exempt': 11322}
loan_term -> {'Exempt': 11188, 'NA': 4746}
tract_to_msa_income_percentage -> all values convert
ffiec_msa_md_median_family_income -> all values convert


The only values that cannot be converted are three codes: "NA", "Exempt" and empty text, so no other text hides in the numeric columns. The loan amount and the two census tract columns convert completely. Every other numeric column has "NA" codes, from 4,746 in the loan term to 277,487 in the rate spread. "Exempt" appears in every one of those columns except the income, with between 11,188 and 11,322 records per column. Empty text appears only in the discount points (96,519) and the lender credits (148,810), so these two columns write a missing value in a third way. The three codes may not mean the same thing, so they are kept apart when the columns are converted.

In [ ]:
# Record, for each numeric column, which code each value came from: "NA", "Exempt", empty text or a real value
CODES = ["NA", "Exempt", ""]
code_of = pd.DataFrame({col: df[col].where(df[col].isin(CODES), "value") for col in NUM_COLS})

# Convert the numeric columns: the three codes become missing values, any other bad value raises an error
for col in NUM_COLS:
    df[col] = pd.to_numeric(df[col].where(~df[col].isin(CODES)))

# Show the type and the number of missing values of each converted column, split by the code they came from
summary = pd.DataFrame({"type": df[NUM_COLS].dtypes.astype(str),
                        "missing": df[NUM_COLS].isna().sum()})
for code, label in [("NA", "NA"), ("Exempt", "Exempt"), ("", "empty")]:
    summary[label] = (code_of == code).sum()
summary

,type,missing,NA,Exempt,empty
loan_amount,float64,0,0,0,0
property_value,float64,135001,123811,11190,0
loan_to_value_ratio,float64,212316,201125,11191,0
income,float64,92946,92946,0,0
interest_rate,float64,202628,191436,11192,0
rate_spread,float64,288723,277487,11236,0
total_loan_costs,float64,280058,268826,11232,0
origination_charges,float64,277171,265849,11322,0
discount_points,float64,377996,270155,11322,96519
lender_credits,float64,429190,269058,11322,148810


After the conversion every numeric column is a number type, and in each one the missing count equals the sum of the three codes, so only "NA", "Exempt" and empty text became missing values. The loan amount and the two census tract columns are complete. The gaps are large in the fields this project studies: the income is missing in 92,946 records (18.0%), the property value in 135,001 (26.2%), the loan-to-value ratio in 212,316 (41.2%) and the interest rate in 202,628 (39.3%). The pricing columns are missing in more than half of the records, from 277,171 (53.7%) in the origination charges to 429,190 (83.2%) in the lender credits, and the loan term in 15,934 (3.1%). The code behind each missing value is kept in a separate table, so later sections can tell the three kinds apart.

## 2.2 Action taken

Every record has an action taken, the outcome of the application. Unlike a status field with no published definitions, HMDA defines its eight codes in the filing instructions, so each one can be labelled. Counting the records per action shows how many applications became loans, how many ended without one, and how many records are not applications at all. This count also decides where the interest rate can be measured, because a rate exists only when a loan was offered or made.

In [ ]:
# Labels of the eight action codes, as defined in the HMDA filing instructions
ACTION_LABELS = {
    "1": "Loan originated",
    "2": "Approved, not accepted",
    "3": "Application denied",
    "4": "Application withdrawn",
    "5": "File closed for incompleteness",
    "6": "Purchased loan",
    "7": "Preapproval request denied",
    "8": "Preapproval approved, not accepted",
}

# Count the records in each action, in code order, with their share of the table
action_counts = df["action_taken"].value_counts().sort_index()
action_table = pd.DataFrame({"action": action_counts.index.map(ACTION_LABELS),
                             "records": action_counts,
                             "share (%)": action_counts / len(df) * 100})

# Show the number of distinct actions and the table
print(f"Distinct actions: {len(action_table)}")
action_table.style.format({"records": "{:,.0f}", "share (%)": "{:.2f}"})

Distinct actions: 8


,action,records,share (%)
action_taken,,,
1,Loan originated,"242,931",47.10
2,"Approved, not accepted","15,016",2.91
3,Application denied,"86,323",16.74
4,Application withdrawn,"73,654",14.28
5,File closed for incompleteness,"31,222",6.05
6,Purchased loan,"63,831",12.38
7,Preapproval request denied,360,0.07
8,"Preapproval approved, not accepted","2,409",0.47


All eight actions appear. Loans originated are the largest group, 242,931 records (47.10%). Applications that ended without a loan add up to 206,215 (39.98%): denied (86,323; 16.74%), withdrawn (73,654; 14.28%), closed for incompleteness (31,222; 6.05%) and approved but not accepted (15,016; 2.91%). Purchased loans, 63,831 records (12.38%), are loans bought from another institution, not applications made to the reporting lender. Preapproval requests that did not lead to an application are 2,769 (0.54%).

A record is therefore not always an application, and not every record can have a price: an interest rate is expected only where a loan was offered, made or bought (actions 1, 2 and 6). How the actions are grouped for the analysis is decided in Chapter 4.

## 2.3 Missing values

A missing value is not always an error. It can mean "not applicable", "not reported by choice" or "not recorded", and HMDA writes these in different ways. Section 2.1 counted the codes in the numeric columns. The first table here counts them in the columns kept as text. The second table shows how the codes in the key numeric fields depend on the action taken, to test whether "NA" means "not applicable" and where "Exempt" appears.

In [ ]:
# Columns kept as text: every kept column that was not converted to a number
text_cols = [c for c in KEEP if c not in NUM_COLS]

# Count the codes "NA", "Exempt" and empty text in each text column
text_codes = pd.DataFrame({"NA": (df[text_cols] == "NA").sum(),
                           "Exempt": (df[text_cols] == "Exempt").sum(),
                           "empty": (df[text_cols] == "").sum()})
text_codes["share (%)"] = text_codes.sum(axis=1) / len(df) * 100

# Show only the columns with at least one code, from the largest gap to the smallest
text_codes = text_codes[text_codes[["NA", "Exempt", "empty"]].sum(axis=1) > 0]
print(f"Text columns: {len(text_cols)} | with at least one code: {len(text_codes)}")
text_codes.sort_values("share (%)", ascending=False).style.format(
    {"NA": "{:,.0f}", "Exempt": "{:,.0f}", "empty": "{:,.0f}", "share (%)": "{:.1f}"})

Text columns: 20 | with at least one code: 3


,NA,Exempt,empty,share (%)
debt_to_income_ratio,"204,247","11,192",0,41.8
county_code,"5,010",0,0,1.0
conforming_loan_limit,"1,356",0,0,0.3


Of the 20 text columns, only three hold the codes. The debt-to-income ratio has the largest gap, 215,439 records (41.8%): 204,247 "NA" and 11,192 "Exempt", the same counts found in Chapter 1. The county is "NA" in 5,010 records (1.0%), so these records cannot be placed below the state level, and the conforming loan limit is "NA" in 1,356 (0.3%). The other 17 text columns have none of the three codes, which does not mean they are complete: some HMDA fields write "not applicable" or "exempt" as numeric codes, which are checked when each column is examined.

In [ ]:
# Key fields of the project, with the code behind each value: numeric fields from code_of, debt-to-income from the table
KEY = ["income", "debt_to_income_ratio", "property_value", "loan_to_value_ratio", "interest_rate"]
key_codes = code_of[["income", "property_value", "loan_to_value_ratio", "interest_rate"]].copy()
key_codes["debt_to_income_ratio"] = df["debt_to_income_ratio"].where(
    df["debt_to_income_ratio"].isin(["NA", "Exempt"]), "value")
key_codes = key_codes[KEY]

# Share of "NA" and of "Exempt" in each key field, for each action taken
rows = {}
for action, group in key_codes.groupby(df["action_taken"]):
    shares = {"records": len(group)}
    for col in KEY:
        shares[f"{col} NA"] = (group[col] == "NA").mean() * 100
        shares[f"{col} Exempt"] = (group[col] == "Exempt").mean() * 100
    rows[f"{action} {ACTION_LABELS[action]}"] = shares
codes_by_action = pd.DataFrame(rows).T

# Show the table, in percentages
fmt = {c: "{:.1f}" for c in codes_by_action.columns if c != "records"}
codes_by_action.style.format({"records": "{:,.0f}", **fmt})

,records,income NA,income Exempt,debt_to_income_ratio NA,debt_to_income_ratio Exempt,property_value NA,property_value Exempt,loan_to_value_ratio NA,loan_to_value_ratio Exempt,interest_rate NA,interest_rate Exempt
1 Loan originated,"242,931",9.5,0.0,9.8,3.7,2.5,3.7,5.8,3.7,0.5,3.7
"2 Approved, not accepted","15,016",18.1,0.0,20.6,1.6,10.3,1.6,16.9,1.6,1.5,1.6
3 Application denied,"86,323",7.2,0.0,11.3,1.1,13.3,1.1,19.7,1.1,98.9,1.1
4 Application withdrawn,"73,654",8.8,0.0,98.7,1.3,98.7,1.3,98.7,1.3,98.7,1.3
5 File closed for incompleteness,"31,222",14.3,0.0,99.8,0.2,99.8,0.2,99.8,0.2,99.8,0.2
6 Purchased loan,"63,831",78.2,0.0,99.9,0.1,1.5,0.1,99.9,0.1,0.6,0.1
7 Preapproval request denied,360,0.6,0.0,1.1,0.6,0.8,0.6,0.8,0.6,99.4,0.6
"8 Preapproval approved, not accepted","2,409",0.2,0.0,0.8,0.4,1.3,0.4,1.2,0.4,0.6,0.4


Most "NA" codes follow the action taken. Withdrawn applications and files closed for incompleteness have no debt-to-income ratio, property value, loan-to-value ratio or interest rate in at least 98.7% of the records, because the process stopped before these values were set. Denied applications have no interest rate in 98.9%, since no loan was priced, but keep most of the other fields. Purchased loans lack the debt-to-income and loan-to-value ratios in 99.9% and the income in 78.2%, while their interest rate and property value are almost always present. In these groups the gaps are mostly structural: a value is missing because it does not apply.

Loans originated are different. They have an interest rate in almost every record (0.5% "NA"), yet the income is "NA" in 9.5%, the debt-to-income ratio in 9.8%, the loan-to-value ratio in 5.8% and the property value in 2.5%. These gaps are not explained by the action, and they sit in the loans the pricing analysis uses. "Exempt" behaves in another way: it never appears in the income, and within each action it has the same share in the other four fields (3.7% of loans originated, for example). This points to a whole record withheld by its lender, not to a single field left blank.

### Is "Exempt" a field-level or a lender-level code?

If "Exempt" marks a whole record withheld by its lender, the four fields should be exempt together on the same records, and the code should concentrate in a group of lenders. The next cell counts how many of the four fields are exempt in each record, and how the exempt records are spread across lenders.

In [ ]:
# The four key fields that can be "Exempt", and how many of them are exempt in each record
EX_COLS = ["debt_to_income_ratio", "property_value", "loan_to_value_ratio", "interest_rate"]
n_exempt = (key_codes[EX_COLS] == "Exempt").sum(axis=1)
print("Records by number of exempt key fields:")
print(n_exempt.value_counts().sort_index().to_string(), "\n")

# For each lender: records, records with at least one exempt key field, and their share
by_lender = (n_exempt > 0).groupby(df["lei"]).agg(records="size", exempt="sum")
by_lender["share"] = by_lender["exempt"] / by_lender["records"]
exempting = by_lender[by_lender["exempt"] > 0]

# Summary of lenders: how many exempt, and whether they exempt all or only part of their records
print(f"Lenders: {len(by_lender):,} | with at least one exempt record: {len(exempting):,}")
print(f"Exempting lenders that exempt every record: {(exempting['share'] == 1).sum():,} | "
      f"only part of their records: {(exempting['share'] < 1).sum():,}")
print(f"Median records per lender: exempting {exempting['records'].median():,.0f} | "
      f"other {by_lender.loc[by_lender['exempt'] == 0, 'records'].median():,.0f}")

Records by number of exempt key fields:
0    504554
3         3
4     11189 

Lenders: 1,220 | with at least one exempt record: 202
Exempting lenders that exempt every record: 194 | only part of their records: 8
Median records per lender: exempting 6 | other 20


"Exempt" is a lender-level code. In 11,189 of the 11,192 records with an exempt key field, all four fields are exempt together, and in the other 3 records three of them are. Of the 1,220 lenders with records in Georgia, 202 (16.6%) report at least one exempt record, and 194 of these exempt every record they report. Only 8 lenders exempt part of their records. Exempting lenders are small in this table, with a median of 6 records in Georgia against 20 for the other lenders. Lender size here counts Georgia records only, not the lender's activity in other states.

A missing value coded "Exempt" therefore depends on who the lender is, not on the borrower. This is a mechanism that the analysis can observe through the lender identifier, unlike the "NA" codes in loans originated, whose cause the table does not show.

## 2.4 Loan amount and income

The loan amount and the income set the scale of every record, and both are reported in a rounded form. In the public HMDA data the loan amount is the midpoint of a 10,000-dollar interval, and the income is in thousands of dollars. Two kinds of values need a look: values that cannot be real (negative, zero or very small amounts) and extremely large values. The first table profiles both columns, and checks that every loan amount sits at the midpoint of its interval.

In [ ]:
# Profile of a numeric column: counts of impossible values and the main percentiles
def amount_profile(s):
    s = s.dropna()
    return pd.Series({
        "Non-missing": len(s),
        "Negative": (s < 0).sum(),
        "Zero": (s == 0).sum(),
        "Minimum": s.min(),
        "1st percentile": s.quantile(0.01),
        "Median": s.median(),
        "99th percentile": s.quantile(0.99),
        "Maximum": s.max(),
    })

# Profile both columns side by side (loan amount in dollars, income in thousands of dollars)
amount_table = pd.DataFrame({"loan_amount ($)": amount_profile(df["loan_amount"]),
                             "income ($ thousands)": amount_profile(df["income"])})
print(amount_table.to_string(float_format=lambda x: f"{x:,.0f}"), "\n")

# Check that every loan amount is the midpoint of a 10,000-dollar interval (ends in 5,000)
print("Loan amounts ending in 5,000:", (df["loan_amount"] % 10_000 == 5_000).mean() * 100, "%")

                 loan_amount ($)  income ($ thousands)
Non-missing              515,746               422,800
Negative                       0                   367
Zero                           0                 7,213
Minimum                    5,000              -199,334
1st percentile            15,000                     0
Median                   245,000                   100
99th percentile        1,175,000                   950
Maximum          999,000,005,000               780,200 

Loan amounts ending in 5,000: 100.0 %


Every loan amount ends in 5,000, so the rounding to the midpoint of a 10,000-dollar interval holds for the whole table, and no loan amount is negative or zero. The usual range is wide but plausible: the median loan is 245,000 dollars and 99% of loans are at or below 1,175,000. The maximum, 999,000,005,000 dollars, is not a possible mortgage; its repeated nines look like a placeholder typed into the field.

The income has impossible values at both ends. Of the 422,800 records with an income, 367 are negative and 7,213 (1.7%) are zero, and the minimum is -199,334 thousand dollars. The median income is 100 thousand dollars and 99% are at or below 950 thousand, but the maximum is 780,200 thousand, about 780 million dollars. Values this far from the rest need a closer look before they are called errors or extreme cases.

In [ ]:
# Columns that describe each record, to judge whether an extreme value is plausible
SHOW = ["action_taken", "loan_purpose", "occupancy_type", "derived_dwelling_category",
        "loan_amount", "property_value", "income", "lei"]

# Short label for the lender: the last six characters of its identifier
def short(t):
    t = t[SHOW].copy()
    t["lei"] = t["lei"].str[-6:]
    t["action_taken"] = t["action_taken"].map(ACTION_LABELS)
    return t

# The ten largest loan amounts and the ten largest incomes
fmt = lambda x: f"{x:,.0f}"
print("Ten largest loan amounts ($; income in $ thousands)")
print(short(df.nlargest(10, "loan_amount")).to_string(index=False, float_format=fmt), "\n")
print("Ten largest incomes ($ thousands)")
print(short(df.nlargest(10, "income")).to_string(index=False, float_format=fmt))

Ten largest loan amounts ($; income in $ thousands)
      action_taken loan_purpose occupancy_type            derived_dwelling_category     loan_amount  property_value  income    lei
Application denied            4              1 Single Family (1-4 Units):Site-Built 999,000,005,000           5,000       1 JQIY70
Application denied            4              1 Single Family (1-4 Units):Site-Built   9,990,005,000           5,000       1 JQIY70
    Purchased loan           32              1 Single Family (1-4 Units):Site-Built   1,340,505,000       1,915,000     NaN RJ5Y59
    Purchased loan           31              1 Single Family (1-4 Units):Site-Built   1,080,005,000       1,355,000     NaN RJ5Y59
    Purchased loan            1              1 Single Family (1-4 Units):Site-Built     912,005,000       1,155,000     NaN RJ5Y59
    Purchased loan           32              3 Single Family (1-4 Units):Site-Built     750,005,000       8,505,000     NaN RJ5Y59
    Purchased loan            1

The largest values are errors, not large loans, and they repeat by lender. The two largest loan amounts, 999,000,005,000 and 9,990,005,000 dollars, come from the same lender, on denied applications with a property value of 5,000 dollars and an income of 1 thousand dollars, which looks like placeholder values. The next eight, from 630,005,000 to 1,340,505,000 dollars, are all purchased loans from one lender, for properties valued between 825,000 and 8,505,000 dollars. Divided by 1,000, these amounts would be ordinary mortgages for those properties, which points to a unit error by that lender.

The largest incomes show the same pattern. Read in thousands of dollars, they range from 355,591 to 780,200 thousand, that is, between 355 and 780 million dollars a year, for loans between 75,000 and 3,005,000 dollars. Read in dollars instead, most of them would be plausible incomes for these loans. Two lenders account for seven of the ten, and six of those seven are loans originated, so this problem reaches the population the analysis uses.

### How far do the implausible amounts spread?

Two screening rules mark values that no real mortgage would have: a loan amount more than ten times the property value, and an income above 10 million dollars a year (10,000 in thousands). The rules only screen; they do not decide that a value is an error. The next cell counts the records each rule marks and shows which lenders they come from.

In [ ]:
# Screening rules: loan amount above ten times the property value, and income above 10 million dollars
loan_vs_property = df["loan_amount"] > 10 * df["property_value"]
huge_income = df["income"] > 10_000

# For each rule: records marked, lenders involved, and the five lenders with the most marked records
for name, flag in [("Loan > 10 x property value", loan_vs_property), ("Income > $10 million", huge_income)]:
    by_lei = flag.groupby(df["lei"].str[-6:]).agg(records="size", marked="sum")
    by_lei = by_lei[by_lei["marked"] > 0].sort_values("marked", ascending=False)
    by_lei["share of lender (%)"] = by_lei["marked"] / by_lei["records"] * 100
    print(f"{name}: {flag.sum():,} records, {len(by_lei):,} lenders | "
          f"loans originated among them: {(flag & (df['action_taken'] == '1')).sum():,}")
    print(by_lei.head(5).to_string(float_format=lambda x: f"{x:,.1f}"), "\n")

Loan > 10 x property value: 241 records, 22 lenders | loans originated among them: 18
        records  marked  share of lender (%)
lei                                         
RJ5Y59      287     171                 59.6
JLHE19     1663      27                  1.6
OSNR22     3087       7                  0.2
R8CP22       80       6                  7.5
Z6IP44     1435       6                  0.4 

Income > $10 million: 1,026 records, 32 lenders | loans originated among them: 820
        records  marked  share of lender (%)
lei                                         
BUPQ06     1508     961                 63.7
63HS39     1343      14                  1.0
NP7R34     1672       7                  0.4
7VG772     3977       4                  0.1
I7VE83      794       4                  0.5 



Each problem comes mostly from a single lender. The loan amount rule marks 241 records from 22 lenders, and one lender accounts for 171 of them (71.0%), more than half of its own 287 records (59.6%). No other lender has more than 27. The income rule marks 1,026 records from 32 lenders, and one lender accounts for 961 (93.7%), which is 63.7% of its 1,508 records; the next lender has 14. These are not scattered typing mistakes but a reporting problem in two lenders' files, most likely a wrong unit.

The income problem reaches the population the analysis uses: 820 of the 1,026 marked records are loans originated. The loan amount problem barely does, with 18 loans originated, since most of the marked records are purchased loans. Neither rule marks every record of these lenders, so the rest of their file cannot be assumed correct or wrong from this check alone.

## 2.5 Property value, loan-to-value and interest rate

The property value, the loan-to-value ratio and the interest rate complete the numeric fields the analysis needs. The property value is rounded like the loan amount, to the midpoint of a 10,000-dollar interval. The loan-to-value ratio and the interest rate are percentages. The table profiles the three columns with the same measures as section 2.4, and counts loan-to-value ratios above 100%, which mean a loan larger than the property, and interest rates of zero.

In [ ]:
# Profile the three columns side by side, with the function from section 2.4
value_table = pd.DataFrame({"property_value ($)": amount_profile(df["property_value"]),
                            "loan_to_value_ratio (%)": amount_profile(df["loan_to_value_ratio"]),
                            "interest_rate (%)": amount_profile(df["interest_rate"])})
print(value_table.to_string(float_format=lambda x: f"{x:,.2f}"), "\n")

# Check the rounding of the property value: midpoint of a 10,000-dollar interval (ends in 5,000)
pv = df["property_value"].dropna()
print("Property values ending in 5,000:", (pv % 10_000 == 5_000).mean() * 100, "%")

# Count loan-to-value ratios above 100% and above 200%, and interest rates of zero
ltv = df["loan_to_value_ratio"].dropna()
print(f"Loan-to-value above 100%: {(ltv > 100).sum():,} | above 200%: {(ltv > 200).sum():,}")
print(f"Interest rate of zero: {(df['interest_rate'] == 0).sum():,}")

                 property_value ($)  loan_to_value_ratio (%)  interest_rate (%)
Non-missing              380,745.00               303,430.00         313,118.00
Negative                       0.00                     0.00               0.00
Zero                           0.00                     0.00           3,477.00
Minimum                    5,000.00                     0.00               0.00
1st percentile            85,000.00                    10.16               0.00
Median                   375,000.00                    79.99               6.58
99th percentile        1,905,000.00                   103.71              11.90
Maximum            1,605,595,000.00         1,400,000,000.00              15.49 

Property values ending in 5,000: 100.0 %
Loan-to-value above 100%: 11,713 | above 200%: 172
Interest rate of zero: 3,477


Every property value ends in 5,000, so the rounding to the midpoint of a 10,000-dollar interval holds here too, and none is negative or zero. The median property is valued at 375,000 dollars and 99% are at or below 1,905,000, but the maximum is 1,605,595,000 dollars, a value to examine together with the loan amount errors of section 2.4.

The loan-to-value ratio has a median of 79.99% and a 99th percentile of 103.71%. Of the 303,430 records with a ratio, 11,713 (3.9%) are above 100%. A ratio slightly above 100% can be real when fees are added to the loan, but 172 records are above 200%, and the maximum, 1,400,000,000%, is not possible. The smallest ratio is positive but below 0.005%.

The interest rate has no extreme high values: the median is 6.58%, 99% are at or below 11.90%, and the maximum is 15.49%. The problem is at the other end: 3,477 records (1.1% of the 313,118 with a rate) have a rate of zero, which is unusual for a mortgage and is checked next.

### Who reports an interest rate of zero?

A zero rate can be a real product, such as a down-payment assistance loan that charges no interest, or a reporting error. A real product would concentrate in subordinate liens and specific loan purposes, while an error would concentrate in specific lenders. The next cell shows the share of zero rates by lien status, loan type and loan purpose, among the records with a rate, and the lenders that report the most.

In [ ]:
# Records with an interest rate, and whether the rate is zero
rated = df[df["interest_rate"].notna()].assign(zero=lambda t: t["interest_rate"] == 0)

# Share of zero rates by lien status, loan type and loan purpose (codes as in the HMDA filing instructions)
for col in ["lien_status", "loan_type", "loan_purpose"]:
    t = rated.groupby(col).agg(records=("zero", "size"), zero=("zero", "sum"))
    t["zero (%)"] = t["zero"] / t["records"] * 100
    print(t.to_string(formatters={"records": "{:,}".format, "zero": "{:,}".format,
                                  "zero (%)": "{:.1f}".format}), "\n")

# The five lenders with the most zero rates, and the share of their rated records that are zero
by_lei = rated.groupby(rated["lei"].str[-6:]).agg(records=("zero", "size"), zero=("zero", "sum"))
by_lei = by_lei[by_lei["zero"] > 0].sort_values("zero", ascending=False)
by_lei["zero (%)"] = by_lei["zero"] / by_lei["records"] * 100
print(f"Lenders with at least one zero rate: {len(by_lei):,}")
print(by_lei.head(5).to_string(formatters={"records": "{:,}".format, "zero": "{:,}".format,
                                           "zero (%)": "{:.1f}".format}))

            records  zero zero (%)
lien_status                       
1           253,382    67      0.0
2            59,736 3,410      5.7 

          records  zero zero (%)
loan_type                       
1         203,022 3,477      1.7
2          67,718     0      0.0
3          40,328     0      0.0
4           2,050     0      0.0 

             records  zero zero (%)
loan_purpose                       
1            175,262 3,441      2.0
2             18,950     3      0.0
31            41,328     0      0.0
32            53,639     4      0.0
4             22,972    29      0.1
5                967     0      0.0 

Lenders with at least one zero rate: 84
       records zero zero (%)
lei                         
KIZ697   5,697  529      9.3
DU7D76   4,179  416     10.0
HPRY47   2,423  394     16.3
DBR481   4,145  225      5.4
2XR050   2,036  224     11.0


Zero rates belong to one kind of loan. Of the 3,477 zero rates, 3,410 (98.1%) are on subordinate liens, where 5.7% of the rated records have a zero rate, against 67 on first liens. All 3,477 are conventional loans, and 3,441 (99.0%) are for a home purchase. A second loan with no interest, taken out next to the first mortgage to buy a home, matches the description of a down-payment assistance loan.

The lenders confirm this reading. The zero rates come from 84 lenders, and the five with the most account for 1,788 of them (51.4%), but in each of these lenders zero rates are only between 5.4% and 16.3% of the rated records. A reporting error would affect a lender's whole file, as with the incomes in section 2.4; here the same lenders report ordinary rates for most of their loans. The zero rates are therefore most likely real, and they belong to a product different from the first-lien mortgages whose price the analysis studies.

### Do the extreme property values and ratios come from the same lenders?

Section 2.4 found that the impossible loan amounts and incomes concentrate in a few lenders. The next cell applies two more screening rules, a property value above 20 million dollars and a loan-to-value ratio above 200%, and checks whether the records they mark come from the same lenders or overlap with the loan amount rule.

In [ ]:
# Screening rules: property value above 20 million dollars, and loan-to-value ratio above 200%
huge_property = df["property_value"] > 20_000_000
huge_ltv = df["loan_to_value_ratio"] > 200

# Lenders already found in section 2.4 (last six characters of the identifier)
SEEN = {"JQIY70", "RJ5Y59", "BUPQ06"}
lender = df["lei"].str[-6:]

# For each rule: records, lenders, overlap with the loan amount rule and with the lenders seen, top five lenders
for name, flag in [("Property value > $20 million", huge_property), ("Loan-to-value > 200%", huge_ltv)]:
    by_lei = flag.groupby(lender).agg(records="size", marked="sum")
    by_lei = by_lei[by_lei["marked"] > 0].sort_values("marked", ascending=False)
    print(f"{name}: {flag.sum():,} records, {len(by_lei):,} lenders | "
          f"also loan > 10 x property: {(flag & loan_vs_property).sum():,} | "
          f"from lenders seen in 2.4: {(flag & lender.isin(SEEN)).sum():,} | "
          f"loans originated: {(flag & (df['action_taken'] == '1')).sum():,}")
    print(by_lei.head(5).to_string(), "\n")

Property value > $20 million: 198 records, 52 lenders | also loan > 10 x property: 2 | from lenders seen in 2.4: 0 | loans originated: 183
        records  marked
lei                    
8QVZ06       34      22
G3KG47       29      22
GFMZ98       36      19
K09D58       16      14
IHU012       34      10 

Loan-to-value > 200%: 172 records, 37 lenders | also loan > 10 x property: 17 | from lenders seen in 2.4: 0 | loans originated: 17
        records  marked
lei                    
OSNR22     3087      29
Z6IP44     1435      21
KADB04     2398      18
0M1A92     3891      14
MGLV11     6452      13 



The two rules mark different problems, and neither comes from the lenders of section 2.4. Property values above 20 million dollars appear in 198 records from 52 lenders, and 183 of them (92.4%) are loans originated. They concentrate in small lenders where most records are marked, for example 22 of 34 and 22 of 29, and only 2 of them also have a loan above ten times the property value, so the loans are large too. This fits lenders of large properties, such as apartment buildings, more than a unit error, a reading that the dwelling category can confirm.

Loan-to-value ratios above 200% look like scattered errors. They appear in 172 records from 37 lenders, and the lenders with the most are large ones where they are rare, 29 of 3,087 records at the top. Only 17 are loans originated, and 17 also have a loan above ten times the property value. The loan amount rule of section 2.4 and this rule overlap little because most records that the first one marks are purchased loans, which have no loan-to-value ratio.

### What kind of property are the values above 20 million dollars?

If these records are apartment buildings, a property value above 20 million dollars is plausible and the records belong to a different market from single-family homes. If they are single-family homes, the values are extreme cases to flag. The next cell shows the dwelling category and the number of units of the 198 marked records, next to the size of each category in the whole table.

In [ ]:
# Records with a property value above 20 million dollars
big = df[huge_property]

# For the dwelling category and the number of units: marked records, all records, and the marked share
for col in ["derived_dwelling_category", "total_units"]:
    t = pd.DataFrame({"marked": big[col].value_counts(),
                      "all records": df[col].value_counts()}).fillna(0).astype(int)
    t = t[t["marked"] > 0].sort_values("marked", ascending=False)
    t["marked share (%)"] = t["marked"] / t["all records"] * 100
    print(t.to_string(formatters={"marked": "{:,}".format, "all records": "{:,}".format,
                                  "marked share (%)": "{:.1f}".format}), "\n")

                                     marked all records marked share (%)
derived_dwelling_category                                               
Multifamily:Site-Built                  183       1,234             14.8
Single Family (1-4 Units):Site-Built     12     486,581              0.0
Multifamily:Manufactured                  3         122              2.5 

            marked all records marked share (%)
total_units                                    
>149           158         237             66.7
100-149         20          94             21.3
1               11     508,960              0.0
50-99            7         127              5.5
3                1         682              0.1
25-49            1         123              0.8 



Most of the large property values are apartment buildings. Of the 198 marked records, 186 (93.9%) are multifamily properties: 183 site-built and 3 manufactured. By size, 158 have more than 149 units, which is two thirds of all records in that size class (158 of 237), and 20 have between 100 and 149 units. For buildings of this size, a value above 20 million dollars is plausible.

Only 12 marked records are single-family homes, 11 with one unit and 1 with three units, out of 486,581 single-family records. These are the extreme values to flag. The multifamily records belong to a different market from the single-family mortgages the analysis studies, so how they are treated is decided in Chapter 3.

## 2.6 Loan characteristics

The loan characteristics are stored as codes or bands, not as numbers. They define which records belong to the analysis, such as first liens or home purchases, so every value has to be known. Some HMDA fields write "exempt" or "not applicable" as numeric codes, such as 1111, which the count of section 2.3 does not catch. The next cell lists every value of each characteristic with its count.

In [ ]:
# Loan characteristics kept as text (identifiers, geography and the action taken are left out)
CHAR_COLS = ["preapproval", "loan_type", "loan_purpose", "lien_status", "occupancy_type",
             "open-end_line_of_credit", "business_or_commercial_purpose", "reverse_mortgage",
             "construction_method", "total_units", "derived_dwelling_category",
             "derived_loan_product_type", "conforming_loan_limit", "debt_to_income_ratio"]

# For each characteristic: number of distinct values, then every value with its count
for col in CHAR_COLS:
    counts = df[col].value_counts()
    print(f"{col} ({len(counts)} values): {counts.to_dict()}\n")

preapproval (2 values): {'2': 507118, '1': 8628}

loan_type (4 values): {'1': 342812, '2': 107234, '3': 63037, '4': 2663}

loan_purpose (6 values): {'1': 247967, '32': 100864, '31': 74475, '4': 50139, '2': 40960, '5': 1341}

lien_status (2 values): {'1': 406433, '2': 109313}

occupancy_type (3 values): {'1': 465991, '3': 42888, '2': 6867}

open-end_line_of_credit (3 values): {'2': 428705, '1': 75849, '1111': 11192}

business_or_commercial_purpose (3 values): {'2': 473315, '1': 31246, '1111': 11185}

reverse_mortgage (3 values): {'2': 502908, '1111': 11192, '1': 1646}

construction_method (2 values): {'1': 487815, '2': 27931}

total_units (9 values): {'1': 508960, '2': 3845, '4': 903, '5-24': 775, '3': 682, '>149': 237, '50-99': 127, '25-49': 123, '100-149': 94}

derived_dwelling_category (4 values): {'Single Family (1-4 Units):Site-Built': 486581, 'Single Family (1-4 Units):Manufactured': 27809, 'Multifamily:Site-Built': 1234, 'Multifamily:Manufactured': 122}

derived_loan_product_type

Every characteristic holds only the values defined in the HMDA filing instructions, with no stray text. Three fields carry the numeric code 1111, which means "exempt": the open-end line of credit and the reverse mortgage in 11,192 records, and the business or commercial purpose in 11,185. These counts match the 11,192 records withheld by their lender in section 2.3, so the code 1111 is the same lender-level exemption written in another way.

The fields that define the analysis population are large enough for any filter: home purchases are 247,967 records (48.1%), first liens 406,433 (78.8%) and principal residences 465,991 (90.4%). Two fields mix numbers and bands. The number of units is a count from 1 to 4 and a band from 5 upwards. The debt-to-income ratio has 21 values: bands below 36% and above 49% ("<20%", "20%-<30%", "30%-<36%", "50%-60%" and ">60%"), and single values from 36 to 49 in between, plus "NA" and "Exempt". How this ratio becomes a variable the analysis can use is decided in Chapter 4.

The conforming loan limit is "NA" in 1,356 records, the same number as the multifamily records. This table does not show whether they are the same records.

### Are the missing conforming limits the multifamily records?

The conforming loan limit sets which loans can be bought by the government-sponsored enterprises, and it is defined for properties of one to four units. The next cell checks whether the 1,356 records with no limit are the 1,356 multifamily records.

In [ ]:
# Records with no conforming limit, and multifamily records
no_limit = df["conforming_loan_limit"] == "NA"
multifamily = df["derived_dwelling_category"].str.startswith("Multifamily")

# Cross the two: if they are the same records, the off-diagonal cells are zero
pd.crosstab(no_limit.rename("no conforming limit"), multifamily.rename("multifamily"))

multifamily,False,True
no conforming limit,,
False,514390,0
True,0,1356


The 1,356 records with no conforming limit are exactly the 1,356 multifamily records, and no other record lacks the limit. The "NA" here means "not applicable": the limit does not exist for properties of five units or more, so this gap is structural and not a data problem.

## 2.7 Issues log

The table below gathers every finding of the audit in one place. Each row has the area it belongs to, its type, the number of records affected and their share of the 515,746 records; rows that describe a limitation of the source and not a count of records have no number. The types are: **Missing** (no value), **Anomaly** (a value that cannot be right), **Placeholder** (a value that stands for "unknown"), **Extreme value** (possible but far from the rest), **Structure** (a feature of how the table is built) and **Limitation** (something the source does not tell us). Nothing is changed here: Chapter 3 decides what to do with each entry.

In [ ]:
# Issues log: every finding of the audit, with its size computed from the data
n = len(df)
orig = df["action_taken"] == "1"
income, loan, ltv, rate = df["income"], df["loan_amount"], df["loan_to_value_ratio"], df["interest_rate"]
na_in = lambda col: code_of[col] == "NA" if col in code_of else df[col] == "NA"
subordinate = df["lien_status"] == "2"

entries = [
    ("Codes",      "Three codes for a missing value in numeric columns: 'NA', 'Exempt' and empty text",
                   "Structure", None),
    ("Exempt",     "Record withheld by its lender: 'Exempt' in the key fields and 1111 in three characteristics",
                   "Missing", (n_exempt > 0).sum()),
    ("Missing",    "Income 'NA' in loans originated", "Missing", (na_in("income") & orig).sum()),
    ("Missing",    "Debt-to-income ratio 'NA' in loans originated", "Missing",
                   (na_in("debt_to_income_ratio") & orig).sum()),
    ("Missing",    "Property value 'NA' in loans originated", "Missing", (na_in("property_value") & orig).sum()),
    ("Missing",    "Loan-to-value ratio 'NA' in loans originated", "Missing",
                   (na_in("loan_to_value_ratio") & orig).sum()),
    ("Missing",    "Interest rate 'NA' in loans originated", "Missing", (na_in("interest_rate") & orig).sum()),
    ("Missing",    "'NA' in a key field of a record that is not a loan originated (follows the action)",
                   "Structure", ((key_codes == "NA").any(axis=1) & ~orig).sum()),
    ("Missing",    "Pricing fees and credits missing in more than half of the records", "Structure", None),
    ("Missing",    "Empty text in discount points or lender credits", "Missing",
                   ((code_of["discount_points"] == "") | (code_of["lender_credits"] == "")).sum()),
    ("Missing",    "County 'NA'", "Missing", (df["county_code"] == "NA").sum()),
    ("Missing",    "Conforming limit 'NA' (all multifamily records, not applicable)", "Structure",
                   (df["conforming_loan_limit"] == "NA").sum()),
    ("Amounts",    "Loan amounts of 9,990,005,000 and 999,000,005,000 (one lender, denied applications)",
                   "Placeholder", (loan >= 9_990_005_000).sum()),
    ("Amounts",    "Loan amount above ten times the property value (171 from one lender)",
                   "Anomaly", loan_vs_property.sum()),
    ("Amounts",    "Negative income", "Anomaly", (income < 0).sum()),
    ("Amounts",    "Zero income", "Anomaly", (income == 0).sum()),
    ("Amounts",    "Income above 10 million dollars (961 from one lender, likely a unit error)",
                   "Anomaly", huge_income.sum()),
    ("Amounts",    "Loan amount and property value rounded to 10,000 dollars, income in thousands",
                   "Limitation", None),
    ("Property",   "Property value above 20 million dollars in multifamily records", "Structure",
                   (huge_property & multifamily).sum()),
    ("Property",   "Property value above 20 million dollars in single-family records", "Extreme value",
                   (huge_property & ~multifamily).sum()),
    ("Property",   "Loan-to-value ratio above 100% and up to 200%", "Extreme value",
                   ((ltv > 100) & (ltv <= 200)).sum()),
    ("Property",   "Loan-to-value ratio above 200%", "Anomaly", huge_ltv.sum()),
    ("Rate",       "Interest rate of zero on subordinate liens (assistance-type loans)", "Structure",
                   ((rate == 0) & subordinate).sum()),
    ("Rate",       "Interest rate of zero on first liens", "Anomaly", ((rate == 0) & ~subordinate).sum()),
    ("Bands",      "Debt-to-income ratio in bands below 36% and above 49%, single values in between",
                   "Structure", None),
    ("Bands",      "Number of units as a count from 1 to 4 and a band from 5 upwards", "Structure", None),
    ("Records",    "Purchased loans, not applications to the reporting lender", "Structure",
                   (df["action_taken"] == "6").sum()),
    ("Records",    "Preapproval requests that did not lead to an application", "Structure",
                   df["action_taken"].isin(["7", "8"]).sum()),
    ("Records",    "No application identifier in the public table", "Limitation", None),
]

issues = pd.DataFrame(entries, columns=["Area", "Finding", "Type", "Rows"])
issues.index = range(1, len(issues) + 1)
issues.index.name = "#"
issues["Share (%)"] = issues["Rows"].apply(lambda r: "" if pd.isna(r) else f"{r / n * 100:.2f}")
issues["Rows"] = issues["Rows"].apply(lambda r: "" if pd.isna(r) else f"{int(r):,}")

pd.set_option("display.max_colwidth", None)
print(f"Issues log: {len(issues)} entries, {n:,} records\n")
print(issues.to_string())

Issues log: 29 entries, 515,746 records

        Area                                                                                      Finding           Type     Rows Share (%)
#                                                                                                                                          
1      Codes            Three codes for a missing value in numeric columns: 'NA', 'Exempt' and empty text      Structure                   
2     Exempt  Record withheld by its lender: 'Exempt' in the key fields and 1111 in three characteristics        Missing   11,192      2.17
3    Missing                                                              Income 'NA' in loans originated        Missing   23,118      4.48
4    Missing                                                Debt-to-income ratio 'NA' in loans originated        Missing   23,767      4.61
5    Missing                                                      Property value 'NA' in loans originated        Missin

The log has 29 entries: 8 Missing, 6 Anomaly, 1 Placeholder, 2 Extreme value, 10 Structure and 2 Limitation. A share of 0.00 means fewer than 0.005% of the records. The largest entries are features of how the table is built and not errors: 258,022 records (50.03%) have "NA" in a key field because of their action, 193,020 (37.43%) have empty text in the discount points or the lender credits, and 63,831 (12.38%) are purchased loans and not applications.

The gaps that matter for the analysis are the ones in loans originated: the debt-to-income ratio is "NA" in 23,767 records (4.61% of the table), the income in 23,118 (4.48%), the loan-to-value ratio in 13,987 (2.71%), the property value in 5,957 (1.16%) and the interest rate in 1,212 (0.23%). On top of these, 11,192 records (2.17%) are withheld by their lender. The six Anomaly entries affect at most 9,086 records (1.76%), fewer if they overlap, and 7,213 of them are zero incomes; the other five add up to 1,873 (0.36%). The one Placeholder is the two loan amounts typed with repeated nines. The 2 Limitation entries are not counts of records but things the source does not provide: the exact amounts behind the rounding, and an identifier for each application.

## Summary of Chapter 2: The audit

**Question:** What does the table contain, and what is wrong with it?

**Answer:** The table holds one record per application, purchased loan or preapproval request, and it can support the analysis. Most gaps follow the outcome of the application, and the values that cannot be right are few and concentrate in a small number of lenders. The audit logged 29 findings. The gaps that matter are the ones in loans originated, where the income and the debt-to-income ratio are missing in close to one record in ten, and the records withheld by their lender. Nothing was changed: Chapter 3 decides what to do with each finding.

**What the evidence shows:**
- **Structure.** 515,746 records and 33 columns. In the numeric columns only three codes block the conversion: "NA", "Exempt" and empty text. The code behind each missing value is kept in a separate table.
- **Action taken.** Loans originated are 242,931 records (47.10%), applications that ended without a loan 206,215 (39.98%), purchased loans 63,831 (12.38%) and preapproval requests 2,769 (0.54%).
- **Missing values.** Most "NA" codes follow the action: withdrawn applications and closed files lack the debt-to-income ratio, the property value, the loan-to-value ratio and the interest rate in at least 98.7% of the records, and denied applications lack the rate in 98.9%. In loans originated, the income is missing in 9.5%, the debt-to-income ratio in 9.8%, the loan-to-value ratio in 5.8%, the property value in 2.5% and the rate in 0.5%.
- **Exempt.** 11,192 records are withheld by their lender, with all four key fields exempt together in 11,189 of them. 202 of the 1,220 lenders exempt at least one record, and 194 of them exempt every record. The code 1111 in three characteristics is the same exemption.
- **Loan amount and income.** Both are rounded, the loan amount to the midpoint of a 10,000-dollar interval and the income to thousands of dollars. Two loan amounts are placeholders with repeated nines, and 241 loans are above ten times the property value, 171 of them from one lender. The income is negative in 367 records and zero in 7,213, and 1,026 incomes are above 10 million dollars, 961 of them from one lender, which points to a unit error. 820 of these 1,026 records are loans originated.
- **Property value, loan-to-value and rate.** 198 property values are above 20 million dollars, and 186 of them are multifamily buildings. 172 loan-to-value ratios are above 200%. The interest rate has no extreme high values (median 6.58%, maximum 15.49%), and 3,410 of its 3,477 zeros are on subordinate liens, which fits loans that charge no interest, such as down-payment assistance.
- **Loan characteristics.** Every field holds only defined values. The debt-to-income ratio mixes bands and single values, and the number of units mixes counts and bands. The 1,356 missing conforming limits are exactly the multifamily records, where the limit does not apply.

# Chapter 3: The decisions

**Question:** What is done with each finding of the audit, and how is it done without losing the raw data?

**Why it matters:** Every choice made here changes what the analysis can say. Setting a value to missing, keeping it or flagging it leads to different results, so each choice is written down, applied in code and checked, and a reader can disagree with any of them and trace its effect.

Five rules govern all the decisions:
- **No raw value is overwritten.** When a value is not accepted, a new column ending in `_clean` holds the corrected version, and the original column stays untouched.
- **No row is removed.** A record with one problem may be perfectly good in every other variable, so problems are marked, not deleted. Which records the analysis uses is defined in Chapter 4.
- **Suspicious records that stay unchanged get a flag.** Flag columns start with `flag_` and let the analysis include or exclude those records on purpose.
- **Nothing is imputed.** A missing value stays missing. Handling missing values is the subject of the second notebook.
- **The reason for each missing value is kept.** For the key fields, a column ending in `_code` records whether a value is present, "NA", "Exempt" or empty, because the second notebook treats each reason differently.

## 3.1 Decision for each finding

The first column gives the entry numbers of the issues log in section 2.7, and the last one the column that carries the result.

| Log # | Finding | Decision | Result |
|---|---|---|---|
| 1 and 2 | Three codes for a missing value; records withheld by their lender | Missing values stay missing. For the five key fields, the reason is kept: present, "NA", "Exempt" or empty | `income_code`, `debt_to_income_ratio_code`, `property_value_code`, `loan_to_value_ratio_code`, `interest_rate_code` |
| 3 to 7 | Key fields "NA" in loans originated | Kept as missing, with no imputation. Handling them is the subject of the second notebook | `_code` columns |
| 8 | "NA" that follows the action | Kept. These records leave the analysis through the population defined in Chapter 4 | No change |
| 9 and 10 | Pricing fees and credits mostly missing; empty text in points and credits | Kept as missing. The analysis does not use these columns | No change |
| 11 | County "NA" | Kept as missing. These records can be used at state level only | No change |
| 12 | Conforming limit "NA" in multifamily records | Kept: the limit does not apply to these properties | No change |
| 13 and 14 | Placeholder loan amounts; loan above ten times the property value | Not accepted as real and set to missing in the clean column. The two placeholders also meet the second rule | `loan_amount_clean` |
| 15 | Negative income | Set to missing in the clean column | `income_clean` |
| 16 | Zero income | Kept as recorded and flagged, because a zero is unusual but not clearly impossible | `flag_zero_income` |
| 17 | Income above 10 million dollars | Set to missing in the clean column. Every record of the lender with most of these values is flagged, because its other incomes may carry the same unit error | `income_clean`, `flag_income_unit_lender` |
| 18 | Rounded loan amount and property value, income in thousands | Kept and described in the data dictionary | No change |
| 19 | Property value above 20 million dollars in multifamily records | Kept. Multifamily records leave the analysis through the population defined in Chapter 4 | No change |
| 20 | Property value above 20 million dollars in single-family records | Kept as recorded and flagged as extreme values | `flag_large_property_sf` |
| 21 | Loan-to-value ratio above 100% and up to 200% | Kept: a ratio slightly above 100% can be real | No change |
| 22 | Loan-to-value ratio above 200% | Set to missing in the clean column | `loan_to_value_ratio_clean` |
| 23 | Zero rate on subordinate liens | Kept: these are real loans. Subordinate liens leave the analysis through the population defined in Chapter 4 | No change |
| 24 | Zero rate on first liens | Set to missing in the clean column | `interest_rate_clean` |
| 25 and 26 | Debt-to-income ratio and number of units in bands | Converted into variables the analysis can use in Chapter 4 | Chapter 4 |
| 27 and 28 | Purchased loans; preapproval requests | Kept. They leave the analysis through the population defined in Chapter 4 | No change |
| 29 | No application identifier | Kept and described in the limitations. The completeness of the copy rests on the match with the Data Browser total | No change |

## 3.2 Applying the decisions

One cell applies all the decisions to a copy of the table called `data`. The raw columns are not touched: each corrected version goes into a new `_clean` column, each flag into a `flag_` column, and the reason for each missing key value into a `_code` column. The cell ends with a

In [ ]:
# Apply the decisions to a copy of the table: raw columns stay untouched
data = df.copy()
income, loan, pv = data["income"], data["loan_amount"], data["property_value"]
ltv, rate = data["loan_to_value_ratio"], data["interest_rate"]

# Log 1 to 7: reason for each missing key value (present, "NA", "Exempt" or empty), from section 2.3
for col in KEY:
    data[f"{col}_code"] = key_codes[col].replace({"value": "present", "": "empty"})

# Log 13 and 14: loan amounts above ten times the property value become missing (includes the two placeholders)
data["loan_amount_clean"] = loan.mask(loan > 10 * pv)

# Log 15 and 17: negative incomes and incomes above 10 million dollars become missing
data["income_clean"] = income.mask((income < 0) | (income > 10_000))

# Log 16: zero incomes stay as recorded, flagged
data["flag_zero_income"] = income == 0

# Log 17: every record of the lender with the most incomes above 10 million dollars, flagged
unit_lender = data.loc[income > 10_000, "lei"].value_counts().idxmax()
data["flag_income_unit_lender"] = data["lei"] == unit_lender

# Log 20: single-family properties valued above 20 million dollars stay as recorded, flagged
multifamily = data["derived_dwelling_category"].str.startswith("Multifamily")
data["flag_large_property_sf"] = (pv > 20_000_000) & ~multifamily

# Log 22: loan-to-value ratios above 200% become missing
data["loan_to_value_ratio_clean"] = ltv.mask(ltv > 200)

# Log 24: zero interest rates on first liens become missing
data["interest_rate_clean"] = rate.mask((rate == 0) & (data["lien_status"] == "1"))

# Records changed by each clean column, flagged by each flag, and coded as missing in each code column
def n_changed(raw, new):
    return int((~((raw == new) | (raw.isna() & new.isna()))).sum())

applied = pd.Series({
    **{f"{c}_clean": n_changed(data[c], data[f"{c}_clean"])
       for c in ["loan_amount", "income", "loan_to_value_ratio", "interest_rate"]},
    **{f: int(data[f].sum()) for f in ["flag_zero_income", "flag_income_unit_lender", "flag_large_property_sf"]},
    **{f"{c}_code (not present)": int((data[f"{c}_code"] != "present").sum()) for c in KEY},
}, name="Records changed, flagged or coded")

print(f"Records in data: {len(data):,}   Columns: {data.shape[1]}\n")
print(applied.to_frame().to_string(formatters={"Records changed, flagged or coded": "{:,}".format}))

Records in data: 515,746   Columns: 45

                                        Records changed, flagged or coded
loan_amount_clean                                                     241
income_clean                                                        1,393
loan_to_value_ratio_clean                                             172
interest_rate_clean                                                    67
flag_zero_income                                                    7,213
flag_income_unit_lender                                             1,508
flag_large_property_sf                                                 12
income_code (not present)                                          92,946
debt_to_income_ratio_code (not present)                           215,439
property_value_code (not present)                                 135,001
loan_to_value_ratio_code (not present)                            212,316
interest_rate_code (not present)                                  202,62

Every new column changes exactly the number of records found in the audit: 241 loan amounts (including the two placeholders), 1,393 incomes (the 367 negative and the 1,026 above 10 million dollars), 172 loan-to-value ratios and 67 interest rates of zero on first liens. The flags mark 7,213 zero incomes, the 1,508 records of the lender with most incomes above 10 million dollars and 12 single-family properties valued above 20 million dollars. The code columns record the same missing values counted in the audit: 92,946 incomes, 215,439 debt-to-income ratios, 135,001 property values, 212,316 loan-to-value ratios and 202,628 interest rates. The table now has 45 columns, the 33 original ones plus 4 clean columns, 3 flags and 5 code columns, and still 515,746 records.

## 3.3 Check

Four things must hold after the decisions: no record was added or removed, no raw column changed, every new column has the size expected from the audit, and each code column agrees with the values it describes. The next cell tests them, and each test prints PASS or FAIL.

In [ ]:
# Check: records, raw columns, sizes of the changes and consistency of the code columns
tests = []

# 1. No record added or removed
tests.append(("Same number of records as the snapshot", len(data) == len(snap)))

# 2. Raw columns identical to the audited table
tests.append(("All raw columns unchanged", all(data[col].equals(df[col]) for col in df.columns)))

# 3. Sizes of the changes agree with the issues log
tests.append(("loan_amount_clean: 241 loans above ten times the property value",
              applied["loan_amount_clean"] == loan_vs_property.sum() == 241))
tests.append(("income_clean: 367 negative + 1,026 above 10 million = 1,393",
              applied["income_clean"] == (df["income"] < 0).sum() + huge_income.sum() == 1_393))
tests.append(("loan_to_value_ratio_clean: 172 ratios above 200%",
              applied["loan_to_value_ratio_clean"] == huge_ltv.sum() == 172))
tests.append(("interest_rate_clean: 67 zero rates on first liens",
              applied["interest_rate_clean"] == 67))
tests.append(("flag_income_unit_lender holds 961 of the 1,026 incomes above 10 million",
              (data["flag_income_unit_lender"] & huge_income).sum() == 961))

# 4. Each code column agrees with its raw column: "present" exactly where a value exists
codes_ok = all(((data[f"{c}_code"] == "present") == data[c].notna()).all() for c in KEY if c in NUM_COLS)
dti_ok = ((data["debt_to_income_ratio_code"] == "present")
          == ~data["debt_to_income_ratio"].isin(["NA", "Exempt"])).all()
tests.append(("Code columns agree with the values they describe", codes_ok and dti_ok))

# 5. No clean value beyond its limit, checked only where the values exist
both = data["loan_amount_clean"].notna() & data["property_value"].notna()
tests.append(("loan_amount_clean: none above ten times the property value",
              (data.loc[both, "loan_amount_clean"] <= 10 * data.loc[both, "property_value"]).all()))
tests.append(("income_clean: all between 0 and 10 million dollars",
              data["income_clean"].dropna().between(0, 10_000).all()))
tests.append(("loan_to_value_ratio_clean: none above 200%",
              (data["loan_to_value_ratio_clean"].dropna() <= 200).all()))
tests.append(("interest_rate_clean: no zero rate on first liens",
              not ((data["interest_rate_clean"] == 0) & (data["lien_status"] == "1")).any()))

for label, ok in tests:
    print(f"{'PASS' if ok else 'FAIL'}  {label}")
print(f"\n{sum(bool(ok) for _, ok in tests)} of {len(tests)} tests passed")

PASS  Same number of records as the snapshot
PASS  All raw columns unchanged
PASS  loan_amount_clean: 241 loans above ten times the property value
PASS  income_clean: 367 negative + 1,026 above 10 million = 1,393
PASS  loan_to_value_ratio_clean: 172 ratios above 200%
PASS  interest_rate_clean: 67 zero rates on first liens
PASS  flag_income_unit_lender holds 961 of the 1,026 incomes above 10 million
PASS  Code columns agree with the values they describe
PASS  loan_amount_clean: none above ten times the property value
PASS  income_clean: all between 0 and 10 million dollars
PASS  loan_to_value_ratio_clean: none above 200%
PASS  interest_rate_clean: no zero rate on first liens

12 of 12 tests passed


All 12 tests pass. The table still has the same records as the snapshot, every raw column is identical to the table that was audited, and each new column changes the number of records expected from the issues log. Every code column marks a value as present exactly where the value exists, and no value beyond its limit survives in the clean columns. The decisions of Chapter 3 are therefore applied as written.

## Summary of Chapter 3: The decisions

**Question:** What is done with each finding of the audit, and how is it done without losing the raw data?

**Answer:** Each of the 29 findings has a written decision, applied in one cell to a copy of the table. The result is 12 new columns, 4 corrected `_clean` columns, 3 flags and 5 `_code` columns, while every raw column is intact, no record was removed and nothing was imputed. Twelve tests confirm that the changes have the sizes found in the audit.

**What the evidence shows:**
- **Set to missing in a clean column.** 241 loan amounts above ten times the property value, including the two placeholders; 1,393 incomes, 367 negative and 1,026 above 10 million dollars; 172 loan-to-value ratios above 200%; and 67 zero interest rates on first liens.
- **Kept and flagged.** 7,213 zero incomes, the 1,508 records of the lender whose incomes most likely carry a unit error, and 12 single-family properties valued above 20 million dollars.
- **Reason for each missing value kept.** The five key fields have a code column that records whether each value is present, "NA" or "Exempt", so the second notebook can treat each reason on its own.
- **Kept as recorded.** The missing values, with no imputation, and the loan-to-value ratios above 100% and up to 200%, which can be real. Multifamily records, zero rates on subordinate liens, purchased loans and preapproval requests are also kept, and leave the analysis through the population defined in Chapter 4.
- **Left for Chapter 4.** The conversion of the debt-to-income ratio and the number of units from bands, and the definition of the records the analysis uses.

# Chapter 4: The new variables

**Question:** Which variables does the analysis need that the table does not have, and how are they defined?

**Why it matters:** The analysis of mortgage pricing needs a debt-to-income ratio it can measure, a way to tell manufactured homes apart, and a clear definition of the loans it studies. Each one is defined once, here, so every later analysis uses the same definition and a reader can check it.

## 4.1 Variables created

| Section | Variable | Definition |
|---|---|---|
| 4.2 | `dti_band` | The debt-to-income ratio in seven ordered bands, with one scheme for every record |
| 4.2 | `dti_mid` | The debt-to-income ratio as a number: single values as recorded, closed bands at their midpoint, open bands at an assumed value |
| 4.3 | `manufactured` | True for manufactured homes, False for site-built homes |
| 4.3 | `in_analysis` | True for the loans the analysis studies: originated, first lien, home purchase, single-family, principal residence, and not a reverse mortgage, an open-end line of credit or a business loan |

## 4.2 Debt-to-income ratio

HMDA reports the ratio in bands below 36% and from 50% upwards, and as a single value from 36 to 49 (section 2.6). `dti_band` puts every record in one of seven ordered bands, splitting the single values at 43%. `dti_mid` turns the ratio into a number: the single values stay as recorded, the closed bands take their midpoint (25 for 20%-<30%, 33 for 30%-<36% and 55 for 50%-60%), and the two open bands take an assumed value, 15 for <20% and 65 for >60%. These two values are assumptions, written down so they can be changed. "NA" and "Exempt" stay missing in both, and the reason remains in `debt_to_income_ratio_code`.

In [ ]:
# Raw value -> band and number (single values from 36 to 49 are added below)
BAND_OF = {"<20%": "<20%", "20%-<30%": "20%-<30%", "30%-<36%": "30%-<36%",
           "50%-60%": "50%-60%", ">60%": ">60%"}
MID_OF = {"<20%": 15, "20%-<30%": 25, "30%-<36%": 33, "50%-60%": 55, ">60%": 65}
for v in range(36, 50):
    BAND_OF[str(v)] = "36%-<43%" if v < 43 else "43%-<50%"
    MID_OF[str(v)] = v

# Seven ordered bands, and the number; "NA" and "Exempt" stay missing
BAND_ORDER = ["<20%", "20%-<30%", "30%-<36%", "36%-<43%", "43%-<50%", "50%-60%", ">60%"]
dti = data["debt_to_income_ratio"]
data["dti_band"] = pd.Categorical(dti.map(BAND_OF), categories=BAND_ORDER, ordered=True)
data["dti_mid"] = dti.map(MID_OF).astype(float)

# Every raw value with its count, band and number, to check the mapping
mapping = (data.groupby("debt_to_income_ratio", observed=True)
               .agg(records=("dti_mid", "size"), dti_band=("dti_band", "first"), dti_mid=("dti_mid", "first"))
               .sort_values("dti_mid", na_position="last"))
print(mapping.to_string(formatters={"records": "{:,}".format}))
print(f"\nRecords per band:\n{data['dti_band'].value_counts(sort=False).to_string()}")
print(f"\nMissing: dti_band {data['dti_band'].isna().sum():,} | dti_mid {data['dti_mid'].isna().sum():,}")

                     records  dti_band  dti_mid
debt_to_income_ratio                           
<20%                  17,399      <20%     15.0
20%-<30%              38,712  20%-<30%     25.0
30%-<36%              40,361  30%-<36%     33.0
36                     7,835  36%-<43%     36.0
37                     8,308  36%-<43%     37.0
38                     8,854  36%-<43%     38.0
39                     9,353  36%-<43%     39.0
40                     9,749  36%-<43%     40.0
41                    10,072  36%-<43%     41.0
42                    11,627  36%-<43%     42.0
43                    10,847  43%-<50%     43.0
44                    12,942  43%-<50%     44.0
45                     9,650  43%-<50%     45.0
46                     9,387  43%-<50%     46.0
47                     9,434  43%-<50%     47.0
48                    10,368  43%-<50%     48.0
49                    15,394  43%-<50%     49.0
50%-60%               37,116   50%-60%     55.0
>60%                  22,899      >60%  

Every raw value maps to one band and one number, and the 215,439 records with "NA" or "Exempt" stay missing in both variables, so 300,307 records have a ratio. The band from 43% to below 50% is the largest, with 78,022 records (26.0% of those with a ratio), followed by the band from 36% to below 43%, with 65,798. Among the single values, 49 stands out: 15,394 records, more than any other value from 36 to 48, just below the band that starts at 50%.

## 4.3 Manufactured homes and the analysis population

`manufactured` is True when the construction method is a manufactured home and False when the home is site-built. Manufactured homes stay in the analysis population, because they are part of the market, and this column lets the analysis show their effect.

`in_analysis` marks the loans the analysis studies: loans originated, first liens, home purchases, single-family properties and principal residences. Reverse mortgages, open-end lines of credit and business loans are left out. These three conditions exclude only the records coded as "yes", so the records withheld by their lender, coded 1111, stay in the population: they are part of what the second notebook studies. The next cell applies the conditions one at a time and shows how many records remain after each.

In [ ]:
# Manufactured homes (construction method 2) against site-built homes (1)
data["manufactured"] = data["construction_method"] == "2"

# Conditions of the analysis population, applied in this order
CONDITIONS = [
    ("Loan originated",                data["action_taken"] == "1"),
    ("First lien",                     data["lien_status"] == "1"),
    ("Home purchase",                  data["loan_purpose"] == "1"),
    ("Single-family property",         data["derived_dwelling_category"].str.startswith("Single Family")),
    ("Principal residence",            data["occupancy_type"] == "1"),
    ("Not a reverse mortgage",         data["reverse_mortgage"] != "1"),
    ("Not an open-end line of credit", data["open-end_line_of_credit"] != "1"),
    ("Not a business loan",            data["business_or_commercial_purpose"] != "1"),
]

# Records that remain after each condition
keep = pd.Series(True, index=data.index)
steps = []
for label, cond in CONDITIONS:
    keep &= cond
    steps.append((label, int(keep.sum())))
data["in_analysis"] = keep

print(pd.DataFrame(steps, columns=["After condition", "Records"]).to_string(
    index=False, formatters={"Records": "{:,}".format}), "\n")

# What the population holds: manufactured homes, records withheld by their lender and the flagged lender
pop = data[data["in_analysis"]]
print(f"In the population: {len(pop):,} ({len(pop) / len(data) * 100:.1f}% of the table)")
print(f"Manufactured homes: {pop['manufactured'].sum():,} | "
      f"withheld by their lender: {(pop['debt_to_income_ratio_code'] == 'Exempt').sum():,} | "
      f"flagged lender: {pop['flag_income_unit_lender'].sum():,}")

               After condition Records
               Loan originated 242,931
                    First lien 193,818
                 Home purchase 123,566
        Single-family property 123,181
           Principal residence 108,852
        Not a reverse mortgage 108,816
Not an open-end line of credit 108,483
           Not a business loan 108,384 

In the population: 108,384 (21.0% of the table)
Manufactured homes: 5,137 | withheld by their lender: 1,793 | flagged lender: 1,002


The analysis population holds 108,384 loans, 21.0% of the table. The conditions that remove the most records are the first three: of the 242,931 loans originated, 193,818 are first liens and 123,566 of these are home purchases. Keeping principal residences removes another 14,329, and the last three conditions remove 468 together. Within the population, 5,137 loans (4.7%) are for manufactured homes, 1,793 (1.7%) are withheld by their lender and 1,002 (0.9%) come from the lender flagged for a likely unit error in the income.

## 4.4 Check

The new variables must agree with the data they come from. The next cell tests that the records and raw columns did not change, that the two versions of the debt-to-income ratio are missing exactly where the ratio is "NA" or "Exempt" and that each number falls inside its band, that `manufactured` matches the construction method, and that every loan in the population meets all its conditions.

In [ ]:
# Check: the new variables agree with the data they come from
tests = []

# 1. Records and raw columns unchanged
tests.append(("Same number of records as the snapshot", len(data) == len(snap)))
tests.append(("All raw columns unchanged", all(data[col].equals(df[col]) for col in df.columns)))

# 2. Debt-to-income: missing exactly where the code is not "present", and each number inside its band
present = data["debt_to_income_ratio_code"] == "present"
tests.append(("dti_band and dti_mid missing exactly where the ratio is NA or Exempt",
              data["dti_band"].notna().equals(present) and data["dti_mid"].notna().equals(present)))
LIMITS = {"<20%": (0, 20), "20%-<30%": (20, 30), "30%-<36%": (30, 36), "36%-<43%": (36, 43),
          "43%-<50%": (43, 50), "50%-60%": (50, 60.01), ">60%": (60.01, 100)}
inside = all(data.loc[data["dti_band"] == b, "dti_mid"].between(lo, hi, inclusive="left").all()
             for b, (lo, hi) in LIMITS.items())
tests.append(("Every dti_mid falls inside its dti_band", inside))

# 3. manufactured matches the construction method
tests.append(("manufactured is True exactly for construction method 2",
              data["manufactured"].equals(data["construction_method"] == "2")))

# 4. Every loan in the population meets all its conditions, and the size is the one reported
pop = data[data["in_analysis"]]
tests.append(("Every loan in the population meets all eight conditions",
              (pop["action_taken"] == "1").all() and (pop["lien_status"] == "1").all()
              and (pop["loan_purpose"] == "1").all() and (pop["occupancy_type"] == "1").all()
              and pop["derived_dwelling_category"].str.startswith("Single Family").all()
              and (pop[["reverse_mortgage", "open-end_line_of_credit",
                        "business_or_commercial_purpose"]] != "1").all().all()))
tests.append(("The population has 108,384 loans", len(pop) == 108_384))

for label, ok in tests:
    print(f"{'PASS' if ok else 'FAIL'}  {label}")
print(f"\n{sum(bool(ok) for _, ok in tests)} of {len(tests)} tests passed")

PASS  Same number of records as the snapshot
PASS  All raw columns unchanged
PASS  dti_band and dti_mid missing exactly where the ratio is NA or Exempt
PASS  Every dti_mid falls inside its dti_band
PASS  manufactured is True exactly for construction method 2
PASS  Every loan in the population meets all eight conditions
PASS  The population has 108,384 loans

7 of 7 tests passed


All 7 tests pass. The table keeps its 515,746 records and its raw columns, the two versions of the debt-to-income ratio are missing exactly where the ratio is "NA" or "Exempt", and every number falls inside its band. `manufactured` matches the construction method, and every one of the 108,384 loans in the population meets all eight conditions. The new variables are therefore consistent with the data they come from.

## Summary of Chapter 4: The new variables

**Question:** Which variables does the analysis need that the table does not have, and how are they defined?

**Answer:** Four new columns: the debt-to-income ratio as ordered bands and as a number (`dti_band`, `dti_mid`), the type of construction (`manufactured`) and the loans the analysis studies (`in_analysis`). Each is defined once, the assumptions are written down, and seven tests confirm that they agree with the data they come from. The table now has 49 columns and the same 515,746 records.

**What the evidence shows:**
- **Debt-to-income ratio.** 300,307 records have a ratio, in seven ordered bands. The band from 43% to below 50% is the largest, with 78,022 records, and the single value 49 stands out with 15,394. The number takes the midpoint of the closed bands and an assumed value of 15 and 65 for the open ones.
- **Analysis population.** 108,384 loans, 21.0% of the table: loans originated, first liens, home purchases, single-family properties and principal residences, with no reverse mortgages, open-end lines of credit or business loans.
- **What the population keeps on purpose.** 5,137 manufactured homes, 1,793 loans withheld by their lender and 1,002 loans from the lender flagged for a likely unit error in the income, which the second notebook studies.

# Chapter 5: The output

**Question:** What does this notebook deliver, and can the result be trusted and reused?

**Why it matters:** The analysis notebook does not repeat any of the work above. It reads one file, so that file has to be complete, in a known order, checked against the source and described well enough for someone who has not read the notebook to use it correctly.

## 5.1 Columns of the processed dataset

The processed dataset keeps the 33 columns of the snapshot and adds the 16 columns created in Chapters 3 and 4, so it has 49 columns and the 515,746 records of the snapshot. The raw column, its clean version and its code are kept side by side, so any decision can be checked or undone. The columns are placed in groups:

| Group | Columns |
|---|---|
| Identifier and scope | `activity_year`, `lei`, `state_code`, `county_code`, `derived_msa-md` |
| Outcome and population | `action_taken`, `preapproval`, `in_analysis` |
| Loan characteristics | `loan_type`, `loan_purpose`, `lien_status`, `occupancy_type`, `open-end_line_of_credit`, `business_or_commercial_purpose`, `reverse_mortgage`, `construction_method`, `manufactured`, `total_units`, `derived_dwelling_category`, `derived_loan_product_type`, `conforming_loan_limit`, `loan_term` |
| Pricing | `interest_rate`, `interest_rate_clean`, `interest_rate_code`, `rate_spread`, `total_loan_costs`, `origination_charges`, `discount_points`, `lender_credits` |
| Amounts | `loan_amount`, `loan_amount_clean`, `property_value`, `property_value_code`, `loan_to_value_ratio`, `loan_to_value_ratio_clean`, `loan_to_value_ratio_code` |
| Borrower finances | `income`, `income_clean`, `income_code`, `debt_to_income_ratio`, `debt_to_income_ratio_code`, `dti_band`, `dti_mid` |
| Census tract context | `tract_to_msa_income_percentage`, `ffiec_msa_md_median_family_income` |
| Flags | `flag_zero_income`, `flag_income_unit_lender`, `flag_large_property_sf` |

The next cell builds the table in this order and stops with an error if a column is missing, repeated or not listed.

In [ ]:
# Columns of the processed dataset, in groups
PROCESSED_COLS = [
    # identifier and scope
    "activity_year", "lei", "state_code", "county_code", "derived_msa-md",
    # outcome and population
    "action_taken", "preapproval", "in_analysis",
    # loan characteristics
    "loan_type", "loan_purpose", "lien_status", "occupancy_type", "open-end_line_of_credit",
    "business_or_commercial_purpose", "reverse_mortgage", "construction_method", "manufactured",
    "total_units", "derived_dwelling_category", "derived_loan_product_type",
    "conforming_loan_limit", "loan_term",
    # pricing
    "interest_rate", "interest_rate_clean", "interest_rate_code", "rate_spread",
    "total_loan_costs", "origination_charges", "discount_points", "lender_credits",
    # amounts
    "loan_amount", "loan_amount_clean", "property_value", "property_value_code",
    "loan_to_value_ratio", "loan_to_value_ratio_clean", "loan_to_value_ratio_code",
    # borrower finances
    "income", "income_clean", "income_code",
    "debt_to_income_ratio", "debt_to_income_ratio_code", "dti_band", "dti_mid",
    # census tract context
    "tract_to_msa_income_percentage", "ffiec_msa_md_median_family_income",
    # flags
    "flag_zero_income", "flag_income_unit_lender", "flag_large_property_sf",
]

# No repeats, and exactly the columns of the working table
assert len(PROCESSED_COLS) == len(set(PROCESSED_COLS)) == 49
assert set(PROCESSED_COLS) == set(data.columns), set(data.columns) ^ set(PROCESSED_COLS)

processed = data[PROCESSED_COLS].copy()
print(f"processed: {processed.shape[0]:,} records and {processed.shape[1]} columns")
print(f"Columns of the snapshot kept: {len(set(PROCESSED_COLS) & set(snap.columns))} of {snap.shape[1]}")

processed: 515,746 records and 49 columns
Columns of the snapshot kept: 33 of 33


The processed table has the 515,746 records of the snapshot and 49 columns, and all 33 columns of the snapshot are kept. The other 16 are the clean columns, code columns, flags and variables created in Chapters 3 and 4.

## 5.2 Saving the dataset

The table is saved in Parquet, a format that keeps the type of each column (numbers stay numbers, True/False stays True/False and the ordered bands keep their order), is compressed and loads much faster than a CSV. The file name carries the state, the year and the snapshot date. To confirm that nothing is lost in the file, the next cell reads it back and compares it with the table in memory.

In [ ]:
# Save the processed dataset in Parquet and read it back to compare
OUT_FILE = PROCESSED_DIR / f"hmda_{STATE}_{YEAR}_analysis_{SNAP_DATE}.parquet"
processed.to_parquet(OUT_FILE, index=False, compression="zstd")

reloaded = pd.read_parquet(OUT_FILE)
print(f"File: {OUT_FILE}")
print(f"Size: {OUT_FILE.stat().st_size / 1_000_000:,.1f} MB")
print(f"Records and columns: {reloaded.shape[0]:,} and {reloaded.shape[1]}")
print("Same columns, same order:", list(reloaded.columns) == list(processed.columns))
print("Same data types:         ", reloaded.dtypes.equals(processed.dtypes))
print("Same content:            ", processed.reset_index(drop=True).equals(reloaded))

File: data/processed/hmda_GA_2025_analysis_2026-11-09.parquet
Size: 12.0 MB
Records and columns: 515,746 and 49
Same columns, same order: True
Same data types:          True
Same content:             True


The dataset is saved as `data/processed/hmda_GA_2025_analysis_2026-11-09.parquet`, a file of 12.0 MB. Read back, it has the same 515,746 records and 49 columns, in the same order, with the same data types and the same content, so the file holds exactly the table built in this notebook.

## 5.3 Data dictionary

One row per column of the processed dataset. "Missing" counts the empty values of the number columns; in the `_clean` columns it includes the values set to missing in Chapter 3. In the text columns, a missing value is written as a code, and the count of each code is given instead. Codes follow the HMDA filing instructions.

| Column | Type | Definition | Missing |
|---|---|---|---|
| `activity_year` | text | Year of the records (2025) | 0 |
| `lei` | text | Legal Entity Identifier of the reporting lender | 0 |
| `state_code` | text | State (GA) | 0 |
| `county_code` | text | County code | 5,010 "NA" |
| `derived_msa-md` | text | Metropolitan area code, derived by the FFIEC | 0 |
| `action_taken` | text | Outcome: 1 originated, 2 approved not accepted, 3 denied, 4 withdrawn, 5 closed for incompleteness, 6 purchased, 7 preapproval denied, 8 preapproval approved not accepted | 0 |
| `preapproval` | text | 1 preapproval requested, 2 not requested | 0 |
| `in_analysis` | True/False | True for the 108,384 loans the analysis studies (section 4.3) | 0 |
| `loan_type` | text | 1 conventional, 2 FHA, 3 VA, 4 USDA | 0 |
| `loan_purpose` | text | 1 home purchase, 2 home improvement, 31 refinancing, 32 cash-out refinancing, 4 other, 5 not applicable | 0 |
| `lien_status` | text | 1 first lien, 2 subordinate lien | 0 |
| `occupancy_type` | text | 1 principal residence, 2 second residence, 3 investment property | 0 |
| `open-end_line_of_credit` | text | 1 yes, 2 no, 1111 exempt | 11,192 "1111" |
| `business_or_commercial_purpose` | text | 1 primarily for a business purpose, 2 not, 1111 exempt | 11,185 "1111" |
| `reverse_mortgage` | text | 1 yes, 2 no, 1111 exempt | 11,192 "1111" |
| `construction_method` | text | 1 site-built, 2 manufactured home | 0 |
| `manufactured` | True/False | True when `construction_method` is 2 | 0 |
| `total_units` | text | Number of units: a count from 1 to 4, a band from 5 upwards | 0 |
| `derived_dwelling_category` | text | Single family or multifamily, site-built or manufactured | 0 |
| `derived_loan_product_type` | text | Loan type by lien status (8 values) | 0 |
| `conforming_loan_limit` | text | C conforming, NC nonconforming, NA not applicable (multifamily) | 1,356 "NA" |
| `loan_term` | number | Term of the loan, in months | 15,934 |
| `interest_rate` | number | Interest rate, in percent | 202,628 |
| `interest_rate_clean` | number | `interest_rate` with zero rates on first liens set to missing | 202,695 |
| `interest_rate_code` | text | Reason for the rate: present, NA or Exempt | 191,436 "NA", 11,192 "Exempt" |
| `rate_spread` | number | Rate spread over the average prime offer rate, in percentage points | 288,723 |
| `total_loan_costs` | number | Total loan costs, in dollars | 280,058 |
| `origination_charges` | number | Origination charges, in dollars | 277,171 |
| `discount_points` | number | Discount points, in dollars | 377,996 |
| `lender_credits` | number | Lender credits, in dollars | 429,190 |
| `loan_amount` | number | Loan amount in dollars, midpoint of a 10,000-dollar interval | 0 |
| `loan_amount_clean` | number | `loan_amount` with amounts above ten times the property value set to missing | 241 |
| `property_value` | number | Property value in dollars, midpoint of a 10,000-dollar interval | 135,001 |
| `property_value_code` | text | Reason for the value: present, NA or Exempt | 123,811 "NA", 11,190 "Exempt" |
| `loan_to_value_ratio` | number | Loan-to-value ratio, in percent | 212,316 |
| `loan_to_value_ratio_clean` | number | `loan_to_value_ratio` with ratios above 200% set to missing | 212,488 |
| `loan_to_value_ratio_code` | text | Reason for the ratio: present, NA or Exempt | 201,125 "NA", 11,191 "Exempt" |
| `income` | number | Gross annual income, in thousands of dollars | 92,946 |
| `income_clean` | number | `income` with negative incomes and incomes above 10 million dollars set to missing | 94,339 |
| `income_code` | text | Reason for the income: present or NA (income is never exempt) | 92,946 "NA" |
| `debt_to_income_ratio` | text | Debt-to-income ratio as reported: bands below 36% and from 50%, single values from 36 to 49 | 204,247 "NA", 11,192 "Exempt" |
| `debt_to_income_ratio_code` | text | Reason for the ratio: present, NA or Exempt | 204,247 "NA", 11,192 "Exempt" |
| `dti_band` | ordered category | Debt-to-income ratio in seven ordered bands (section 4.2) | 215,439 |
| `dti_mid` | number | Debt-to-income ratio as a number: single values as reported, midpoints of closed bands, 15 and 65 assumed for the open bands | 215,439 |
| `tract_to_msa_income_percentage` | integer | Median family income of the census tract as a percentage of the area's median family income | 0 |
| `ffiec_msa_md_median_family_income` | integer | Median family income of the area, in dollars, estimated by the FFIEC | 0 |
| `flag_zero_income` | True/False | True for the 7,213 records with an income of zero | 0 |
| `flag_income_unit_lender` | True/False | True for the 1,508 records of the lender with most incomes above 10 million dollars | 0 |
| `flag_large_property_sf` | True/False | True for the 12 single-family properties valued above 20 million dollars | 0 |

## 5.4 Conclusions and limitations

### What this notebook delivers

- A dated snapshot of the HMDA records for Georgia in 2025 (2026-11-09): 515,746 records with 33 of the 99 columns, saved in `data/raw`.
- An audit of that table with 29 findings, each with its size, and a decision for every one of them.
- A processed dataset of 49 columns in `data/processed`: the 33 original columns, 4 corrected `_clean` columns, 5 `_code` columns, 3 flags and 4 derived variables. No raw value was overwritten, no record was removed and nothing was imputed.
- A data dictionary for the 49 columns and tests that confirm each step.

### What the analysis has to keep in mind

- The analysis population is the 108,384 loans with `in_analysis` True. The other records stay in the file but describe other outcomes, liens, purposes and properties.
- Missing values are not all the same. The `_code` columns separate values that do not apply ("NA" that follows the action), values the lender chose not to report ("Exempt") and values missing in loans originated, which are the gaps the second notebook handles.
- The income and the loan-to-value ratio have clean versions, and the clean version is the one to use by default. The raw columns remain, so the effect of the errors themselves can be measured.
- The flags mark records that stay unchanged but need care: 7,213 zero incomes, the 1,508 records of the lender with a likely unit error and 12 very large single-family properties.

### Limitations

- **One snapshot.** Lenders can correct and resubmit their filings, and the Data Browser does not state which version of the 2025 data it serves. The frozen file in the repository is what every result in the project refers to.
- **Rounded and banded values.** The loan amount and the property value are rounded to 10,000 dollars, the income to thousands of dollars, and the debt-to-income ratio is partly in bands. `dti_mid` rests on midpoints and on two assumed values, 15 and 65, for the open bands.
- **Records withheld by their lender.** 11,192 records have no debt-to-income ratio, property value, loan-to-value ratio or interest rate because their lender is exempt from reporting them. The values cannot be recovered from the table.
- **Errors found by screening rules.** The rules catch only values far outside any real range. A unit error that produces a plausible value, for example in the other incomes of the flagged lender, cannot be detected in this table.
- **Zero incomes.** The 7,213 zeros are kept because the table does not show whether they mean no income or a value that was not entered.
- **Scope.** The data cover one state and one year, so the results describe the Georgia market in 2025. The demographic fields are left out, so this work is not an assessment of fair lending.
- **No application identifier.** A record cannot be followed across files or years, and the completeness of the copy rests on the match with the Data Browser total.

### Reproducing the results

The notebook downloads the snapshot only if the dated file is not already in `data/raw`, so running it from the first cell with the repository files rebuilds the processed dataset from the same snapshot. The analysis notebook reads only the processed file.

## Summary of Chapter 5: The output

**Question:** What does this notebook deliver, and can the result be trusted and reused?

**Answer:** A processed dataset of 515,746 records and 49 columns, saved in Parquet with the state, the year and the snapshot date in its name, with a data dictionary and a list of limitations. The file read back is identical to the table built here, and each step of the notebook is backed by tests that pass.

**What the evidence shows:**
- **Columns.** 49 columns in eight groups: the 33 columns of the snapshot and 16 new ones (4 clean columns, 5 code columns, 3 flags and 4 derived variables).
- **File.** 12.0 MB in Parquet. Read back, it has the same records, columns, data types and content as the table in memory.
- **Dictionary.** Every column has its type, its definition and its number of missing values or codes.
- **Limits of the data.** 11,192 records are withheld by their lender, several values are rounded or banded, and the data cover one state and one year.

# Closing summary

This notebook turned the raw HMDA records for Georgia in 2025 into a checked analysis dataset.

| | |
|---|---|
| **Source** | HMDA, FFIEC and CFPB Data Browser, Georgia, 2025, snapshot of 2026-11-09 |
| **Records** | 515,746 applications, purchased loans and preapproval requests, with 33 of the 99 columns |
| **Audit** | 29 findings; the 6 anomalies affect at most 1.76% of the records |
| **Largest gaps in loans originated** | Debt-to-income ratio missing in 9.8% and income in 9.5%; 11,192 records in the table withheld by their lender |
| **Set to missing in clean columns** | 241 loan amounts, 1,393 incomes, 172 loan-to-value ratios and 67 interest rates |
| **Flagged, kept unchanged** | 7,213 zero incomes, 1,508 records of the lender with a likely unit error, 12 single-family properties above 20 million dollars |
| **Reason for missing values** | Kept in 5 code columns: present, "NA" or "Exempt" |
| **New variables** | `dti_band`, `dti_mid`, `manufactured`, `in_analysis` |
| **Analysis population** | 108,384 loans: originated, first lien, home purchase, single-family, principal residence |
| **Output** | 49 columns in Parquet, 12.0 MB, with a data dictionary |
| **Checks** | All tests pass (12 in Chapter 3 and 7 in Chapter 4), and the saved file is identical to the table built here |

No raw value was overwritten, no record was removed and nothing was imputed. The analysis of missing data and influence is in the [second notebook](https://github.com/emelylephilibert/mortgage-missing-data-influence/blob/main/02_missing_incomes_and_extreme_loans.ipynb) of the project, which reads the processed file and does not repeat any of this work.